### Phase 9: Information Extraction
**Objective**: Combine document classification and reading order to extract semantic fields into structured JSON.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import sys
import cv2
import json
import torch
import matplotlib.pyplot as plt
from torchvision import models, transforms
from PIL import Image

# Add project root to path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.preprocessing.pipeline import render_pdf_page, to_grayscale, apply_clahe, otsu_threshold
from src.ocr.ocr_engine import extract_text_and_boxes
from src.extraction.extractor import extract_information
from ultralytics import YOLO

# 1. Load Document
pdf_path = "../data/raw/1706.03762v7.pdf" # Change this to an invoice or resume to test extraction!
original_img = render_pdf_page(pdf_path, page_num=0, dpi=300)
gray = to_grayscale(original_img)
preprocessed_img = otsu_threshold(apply_clahe(gray))

# 2. Run OCR & YOLO
yolo_model = YOLO("../models/layout_detection_exp/weights/best.pt")
yolo_results = yolo_model(original_img, verbose=False)
ocr_tokens = extract_text_and_boxes(preprocessed_img)

# 3. Fuse and get Reading Order (Using the functions from Phase 5 & 6)
# Assuming you have your fuse and sort functions saved in a module, or we can just extract all OCR text for now.
# For simplicity in extraction, let's just grab all OCR text:
full_text = [token['text'] for token in ocr_tokens]
print(f"Extracted {len(full_text)} words from OCR.")

# 4. Classify Document
classifier_path = "../models/doc_classifier_resnet18.pth"
checkpoint = torch.load(classifier_path)
class_names = checkpoint['class_names']

model = models.resnet18()
model.fc = torch.nn.Linear(model.fc.in_features, len(class_names))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Prepare image for classifier
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
input_tensor = transform(Image.fromarray(original_img)).unsqueeze(0)

with torch.no_grad():
    outputs = model(input_tensor)
    _, predicted = torch.max(outputs, 1)
    doc_type = class_names[predicted[0]]

print(f"Predicted Document Type: {doc_type}")

# 5. Extract Information
extracted_json = extract_information(doc_type, full_text)

print("\n--- FINAL EXTRACTED JSON ---")
print(json.dumps(extracted_json, indent=4))

Extracted 399 words from OCR.
Predicted Document Type: resume
Overridden Document Type: paper

--- FINAL EXTRACTED JSON ---
{
    "document_type": "paper"
}
